<div style="background:linear-gradient(135deg,#0F3D6E 0%,#2176AE 100%);padding:40px 36px 32px 36px;border-radius:10px;margin-bottom:8px;">
  <p style="color:#C8DEF5;font-size:13px;margin:0 0 6px 0;letter-spacing:2px;">CURSO 8 · MÓDULO 1 · CLASE 2</p>
  <h1 style="color:white;font-size:36px;margin:0 0 10px 0;font-weight:700;">Ejercicios: Eigenvalues, MVN y Distribución de β̂</h1>
  <p style="color:#C8DEF5;font-size:16px;margin:0 0 24px 0;font-style:italic;">Práctica guiada — intentá antes de ver la solución</p>
  <hr style="border-color:#5BA4CF;margin:0 0 20px 0;">
  <p style="color:#EAF2FB;font-size:13px;margin:0;">📌 <strong>Docente:</strong> Josef Rodriguez &nbsp;·&nbsp; 🟢 Básico · 🟡 Intermedio · 🔴 Avanzado</p>
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'figure.dpi':110,'font.size':11,
                     'axes.spines.top':False,'axes.spines.right':False})
SEED = 42; np.random.seed(SEED)
print('✅ Setup listo')

---
## 🟢 Ejercicio 1 — Eigendecomposición y reconstrucción

Dada la matriz A:
1. Calcular eigenvalues y eigenvectors con `eigh`
2. Verificar que A = VΛVᵀ
3. Calcular tr(A) y det(A) usando solo los eigenvalues
4. ¿Es A definida positiva? ¿Invertible?
5. Calcular A⁻¹ usando la fórmula: A⁻¹ = V Λ⁻¹ Vᵀ

In [ ]:
A = np.array([[6, 2, 1],
              [2, 4, 2],
              [1, 2, 5]], dtype=float)

# --- Tu código aquí ---


In [ ]:
# ✅ SOLUCIÓN
A = np.array([[6,2,1],[2,4,2],[1,2,5]], dtype=float)
eigvals, V = np.linalg.eigh(A)
Lambda     = np.diag(eigvals)

print(f'Eigenvalues: {eigvals.round(4)}')
print(f'Todos > 0 → def. positiva: {np.all(eigvals > 0)}')
print(f'\nA = VΛVᵀ: {np.allclose(A, V @ Lambda @ V.T)}')
print(f'tr(A) = Σλᵢ = {eigvals.sum():.4f}  vs  np.trace = {np.trace(A):.4f}  ✓')
print(f'det(A) = Πλᵢ = {np.prod(eigvals):.4f}  vs  np.det = {np.linalg.det(A):.4f}  ✓')

# Inversa vía eigendecomposición
A_inv_eigen = V @ np.diag(1/eigvals) @ V.T
A_inv_numpy = np.linalg.inv(A)
print(f'\nA⁻¹ via eigendecomp == np.linalg.inv: {np.allclose(A_inv_eigen, A_inv_numpy)}')
print('A⁻¹ ='); print(A_inv_numpy.round(4))

---
## 🟢 Ejercicio 2 — Eigenvalues de XᵀX: lectura del dataset

Dado el dataset generado abajo:
1. Construir X con intercepto y estandarizar
2. Calcular eigenvalues de XᵀX
3. ¿Cuántos eigenvalues son 'pequeños' (< 1)? ¿Qué indica?
4. Calcular el número de condición como ratio max/min eigenvalue
5. Visualizar los eigenvalues en un gráfico de barras

In [ ]:
np.random.seed(SEED)
n, p = 150, 5
X_raw = np.random.randn(n, p)
X_raw[:, 4] = X_raw[:, 1] * 0.9 + np.random.randn(n) * 0.2  # alta correlación

# --- Tu código aquí ---


In [ ]:
# ✅ SOLUCIÓN
np.random.seed(SEED)
n, p = 150, 5
X_raw = np.random.randn(n, p)
X_raw[:, 4] = X_raw[:, 1] * 0.9 + np.random.randn(n) * 0.2

X = np.column_stack([np.ones(n), StandardScaler().fit_transform(X_raw)])
XtX = X.T @ X
eigvals = np.linalg.eigvalsh(XtX)

print(f'Eigenvalues de XᵀX: {eigvals.round(2)}')
print(f'Eigenvalue mín: {eigvals.min():.4f}')
print(f'Eigenvalue máx: {eigvals.max():.4f}')
print(f'Número de condición (max/min): {eigvals.max()/eigvals.min():.1f}')
print(f'cond(XᵀX) = {np.linalg.cond(XtX):.1f}')
print(f'\nEigenvalues < 1 (señal de casi-colinealidad): {(eigvals < 1).sum()}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(eigvals)), sorted(eigvals), color=['#C0392B' if e < 1 else '#2176AE' for e in sorted(eigvals)], alpha=0.85)
ax.axhline(1, color='gray', linestyle='--', lw=1, label='Umbral = 1')
ax.set_xlabel('Componente'); ax.set_ylabel('Eigenvalue'); ax.legend()
ax.set_title('Eigenvalues de XᵀX — rojo = casi-singulares', fontweight='bold')
plt.tight_layout(); plt.show()

---
## 🟡 Ejercicio 3 — Construir y verificar una covarianza Σ

1. Generar 500 muestras de una MVN con μ y Σ dados abajo
2. Estimar μ̂ y Σ̂ empíricamente
3. Verificar que Σ es simétrica, def. positiva e invertible
4. Calcular la matriz de correlaciones R desde Σ̂
5. ¿Qué par de variables tiene mayor correlación?

In [ ]:
mu_dado    = np.array([10.0, -3.0, 5.0, 0.5])
Sigma_dado = np.array([[9.0,  3.0, -1.0,  0.5],
                        [3.0,  4.0,  2.0,  0.0],
                        [-1.0, 2.0, 16.0,  1.5],
                        [0.5,  0.0,  1.5,  1.0]])

# --- Tu código aquí ---


In [ ]:
# ✅ SOLUCIÓN
np.random.seed(SEED)
mu_dado    = np.array([10.0,-3.0,5.0,0.5])
Sigma_dado = np.array([[9,3,-1,0.5],[3,4,2,0],[−1,2,16,1.5],[0.5,0,1.5,1]], dtype=float)
# Fix: sin caracteres especiales
Sigma_dado = np.array([[9.0,3.0,-1.0,0.5],[3.0,4.0,2.0,0.0],[-1.0,2.0,16.0,1.5],[0.5,0.0,1.5,1.0]])

datos = np.random.multivariate_normal(mu_dado, Sigma_dado, 500)
mu_hat    = datos.mean(axis=0)
Sigma_hat = np.cov(datos, rowvar=False)

print(f'μ real:     {mu_dado}')
print(f'μ estimada: {mu_hat.round(3)}')

# Propiedades de Σ_dado
eigv = np.linalg.eigvalsh(Sigma_dado)
print(f'\nΣ simétrica: {np.allclose(Sigma_dado, Sigma_dado.T)}')
print(f'Σ def. positiva: {np.all(eigv > 0)}')
print(f'Σ invertible (det ≠ 0): {abs(np.linalg.det(Sigma_dado)) > 1e-10}')

# Matriz de correlaciones
stds = np.sqrt(np.diag(Sigma_hat))
R    = Sigma_hat / np.outer(stds, stds)
print('\nMatriz de correlaciones R:')
print(R.round(3))

# Par con mayor correlación (fuera de diagonal)
R_abs = np.abs(R)
np.fill_diagonal(R_abs, 0)
i, j = np.unravel_index(R_abs.argmax(), R_abs.shape)
print(f'\nPar con mayor correlación: variables {i} y {j} → ρ={R[i,j]:.3f}')

---
## 🟡 Ejercicio 4 — Simular y verificar la distribución de β̂

1. Fijar X (n=100, p=4 con intercepto) y β_real = [2, -1, 3, 0.5]
2. Simular 3000 muestras de y = Xβ + ε con ε ~ N(0, σ²=4)
3. Calcular β̂ para cada simulación
4. Comparar media empírica de β̂ con β_real (insesgadez)
5. Comparar covarianza empírica con σ²(XᵀX)⁻¹ (teórica)
6. Graficar la distribución de β̂₁ vs su densidad teórica

In [ ]:
np.random.seed(SEED)
n_obs   = 100
beta_r  = np.array([2.0, -1.0, 3.0, 0.5])
sigma2  = 4.0
n_sims  = 3000

X_fijo = np.column_stack([np.ones(n_obs), np.random.randn(n_obs, 3)])

# --- Tu código aquí ---


In [ ]:
# ✅ SOLUCIÓN
np.random.seed(SEED)
n_obs = 100; beta_r = np.array([2.,-1.,3.,0.5]); sigma2 = 4.; n_sims = 3000
X_fijo = np.column_stack([np.ones(n_obs), np.random.randn(n_obs,3)])
XtX_inv = np.linalg.inv(X_fijo.T @ X_fijo)

beta_hats = np.zeros((n_sims, len(beta_r)))
for s in range(n_sims):
    eps = np.random.normal(0, np.sqrt(sigma2), n_obs)
    y_s = X_fijo @ beta_r + eps
    beta_hats[s], *_ = np.linalg.lstsq(X_fijo, y_s, rcond=None)

print('Insesgadez — E[β̂] ≈ β_real:')
for i,(real,emp) in enumerate(zip(beta_r, beta_hats.mean(axis=0))):
    print(f'  β{i}: real={real:.2f}, E[β̂]={emp:.4f}, sesgo={abs(emp-real):.4f}')

Cov_teorica  = sigma2 * XtX_inv
Cov_empirica = np.cov(beta_hats, rowvar=False)
print(f'\n¿Cov empírica ≈ σ²(XᵀX)⁻¹? {np.allclose(Cov_teorica, Cov_empirica, rtol=0.05)}')
print(f'Max diferencia: {np.abs(Cov_teorica - Cov_empirica).max():.4f}')

# Gráfico
fig, ax = plt.subplots(figsize=(7,4))
ax.hist(beta_hats[:,1], bins=60, density=True, color='#2176AE', edgecolor='white', alpha=0.8)
xr = np.linspace(beta_hats[:,1].min(), beta_hats[:,1].max(), 300)
ax.plot(xr, stats.norm.pdf(xr, beta_r[1], np.sqrt(sigma2*XtX_inv[1,1])),
        color='#C0392B', lw=2, label='Teórico N(β₁, σ²(XᵀX)⁻¹₁₁)')
ax.axvline(beta_r[1], color='#0F3D6E', lw=1.5, linestyle='--', label=f'β₁={beta_r[1]}')
ax.set_title('Distribución de β̂₁ (3000 simulaciones)', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

---
## 🔴 Ejercicio 5 — Desafío: diagnóstico completo de Σ

Implementá una función `describe_sigma(Sigma)` que:
1. Verifique simetría, def. positiva, invertibilidad
2. Calcule eigenvalues, tr, det, número de condición
3. Extraiga la matriz de correlaciones R
4. Identifique el par de variables más correlacionadas
5. Imprima un reporte estructurado con alertas si algo falla

In [ ]:
# --- Tu implementación aquí ---
def describe_sigma(Sigma, var_names=None):
    pass

# Tests
S1 = np.array([[4,2,0],[2,3,1],[0,1,2]], dtype=float)           # OK
S2 = np.array([[1,0.95,0],[0.95,1,0],[0,0,1]], dtype=float)    # alta correlación
S3 = np.array([[2,2,0],[2,2,0],[0,0,1]], dtype=float)           # singular


In [ ]:
# ✅ SOLUCIÓN
def describe_sigma(Sigma, var_names=None):
    n_v = Sigma.shape[0]
    names = var_names or [f'x{i}' for i in range(n_v)]
    sep = '─'*50
    print(sep)
    print('  REPORTE DE MATRIZ DE COVARIANZAS Σ')
    print(sep)

    # 1. Propiedades
    sim = np.allclose(Sigma, Sigma.T)
    eigv = np.linalg.eigvalsh(Sigma)
    def_pos = np.all(eigv > 0)
    det_S = np.linalg.det(Sigma)
    invertible = abs(det_S) > 1e-10
    cond = np.linalg.cond(Sigma)

    print(f'  Simétrica:      {sim}  {"✓" if sim else "✗"}')
    print(f'  Def. positiva:  {def_pos}  {"✓" if def_pos else "✗"}')
    print(f'  Invertible:     {invertible}  {"✓" if invertible else "✗"}')
    print(f'  tr(Σ):          {np.trace(Sigma):.4f}')
    print(f'  det(Σ):         {det_S:.4e}')
    print(f'  cond(Σ):        {cond:.2f}  {"✓" if cond < 1000 else "⚠"}')
    print(f'  Eigenvalues:    {eigv.round(4)}')

    # 2. Correlaciones
    stds = np.sqrt(np.diag(Sigma))
    R = Sigma / np.outer(stds, stds)
    np.fill_diagonal(R, 0)
    i,j = np.unravel_index(np.abs(R).argmax(), R.shape)
    print(f'\n  Máx correlación: {names[i]} ↔ {names[j]} = {R[i,j]:.3f}  {"⚠ alta" if abs(R[i,j]) > 0.8 else "✓"}')

    if not def_pos:
        print('  ✗ ALERTA: no def. positiva → no es covarianza válida')
    if cond > 1000:
        print('  ⚠ ALERTA: mal condicionada → alta colinealidad')
    print(sep)

print('\n=== Σ1: covarianza normal ===')
S1 = np.array([[4,2,0],[2,3,1],[0,1,2]], dtype=float)
describe_sigma(S1, ['ingresos','edad','deuda'])

print('\n=== Σ2: alta correlación ===')
S2 = np.array([[1,0.95,0],[0.95,1,0],[0,0,1]], dtype=float)
describe_sigma(S2)

print('\n=== Σ3: singular ===')
S3 = np.array([[2,2,0],[2,2,0],[0,0,1]], dtype=float)
describe_sigma(S3)

---
<div style="background:#0F3D6E;color:white;padding:20px 24px;border-radius:8px;">
<strong>Próxima clase — Viernes</strong><br>
Supuestos Gauss-Markov · Estimación de parámetros · BLUE · Insesgadez<br>
<em>Docente: Josef Rodriguez · Curso 8 · Modelos Estadísticos</em>
</div>